In [ ]:
# Setup & Imports
%pip install -q boruta shap joblib

%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

# Core
import os
import numpy as np
import pandas as pd

# ML & Selection
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    precision_recall_curve, roc_curve, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from boruta import BorutaPy

# Viz
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(context="notebook", style="whitegrid")

# Saving
import joblib

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
#Load Source RDS Files + Enriched CSVs
# Load raw RDS files
try:
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()

    ASV_RDS_PATH  = "1.16S.ASV.profile.original.rds"
    TAX_RDS_PATH  = "1.taxonomy.info.rds"
    META_RDS_PATH = "1.sample.info.original.rds"

    asv_raw  = pandas2ri.rpy2py(robjects.r['readRDS'](ASV_RDS_PATH))
    tax_raw  = pandas2ri.rpy2py(robjects.r['readRDS'](TAX_RDS_PATH))
    meta_raw = pandas2ri.rpy2py(robjects.r['readRDS'](META_RDS_PATH))
    print("Original ASV shape :", asv_raw.shape)
    print("Original Tax shape :", tax_raw.shape)
    print("Original Metadata  :", meta_raw.shape)
except Exception as e:
    tax_raw = None
    print("RDS not loaded (optional). Proceeding with enriched CSVs.")

# Enriched processed files (used for modeling)
META_ENRICHED_PATH = "metadata_enriched.csv"
ASV_CLR_PATH       = "asv_clr_processed.csv"

meta_enriched = pd.read_csv(META_ENRICHED_PATH, index_col=0)
asv_clr = pd.read_csv(ASV_CLR_PATH, index_col=0)

print("Enriched Metadata Shape :", meta_enriched.shape)
print("ASV CLR Matrix Shape    :", asv_clr.shape)

display(meta_enriched.head(2))
display(asv_clr.head(2))


In [ ]:
# Encode Metadata & Merge with ASV CLR → X, y
meta_enc = meta_enriched.copy()

# Sex: F=1, M=0 (if present)
if "Sex" in meta_enc.columns:
    meta_enc["Sex"] = meta_enc["Sex"].map({"F": 1, "M": 0})

# One-hot encode (if present)
encode_cols = [c for c in ["Diet", "Location"] if c in meta_enc.columns]
meta_enc = pd.get_dummies(meta_enc, columns=encode_cols, drop_first=False)

# Convert booleans → int
bool_cols = meta_enc.select_dtypes(include="bool").columns
meta_enc[bool_cols] = meta_enc[bool_cols].astype(int)

# Target label (find column containing "Group (1=HC,2=RA)")
label_col = None
for c in meta_enc.columns:
    if "Group" in c and "HC" in c and "RA" in c:
        label_col = c
        break
if label_col is None:
    raise ValueError("Could not find Group (1=HC,2=RA) column in metadata_enriched.csv")

meta_enc["label"] = meta_enc[label_col].map({1: 0, 2: 1}).astype(int)

# Drop non-ML columns if present
drop_cols = ["Collection date", "SampleID", "Stool characteristics", label_col]
meta_enc = meta_enc.drop(columns=[c for c in drop_cols if c in meta_enc.columns])

# Align ASV rows to metadata index
asv_clr = asv_clr.reindex(meta_enc.index)

# Final matrices
X_full = pd.concat(
    [asv_clr.reset_index(drop=True),
     meta_enc.drop(columns=["label"]).reset_index(drop=True)],
    axis=1
)
y_vec = meta_enc["label"].values.astype(int)

print("X_full:", X_full.shape, "| y:", y_vec.shape)


In [ ]:
#Feature Selection: ExtraTrees Rank → Boruta
print("Initial feature count:", X_full.shape[1])

# Rank by ExtraTrees
et = ExtraTreesClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1)
et.fit(X_full, y_vec)
feat_imp = pd.Series(et.feature_importances_, index=X_full.columns).sort_values(ascending=False)

TOP_K = min(2500, X_full.shape[1])
top_k_features = feat_imp.head(TOP_K).index.tolist()
X_topk = X_full[top_k_features].copy()
print("After ExtraTrees ranking:", X_topk.shape)

# Boruta on top-k
estimator = ExtraTreesClassifier(n_estimators=600, random_state=RANDOM_STATE, n_jobs=-1)
boruta = BorutaPy(estimator=estimator, n_estimators="auto", max_iter=30,
                  random_state=RANDOM_STATE, verbose=2)
boruta.fit(X_topk.values, y_vec)

confirmed_mask = boruta.support_
confirmed_features = X_topk.columns[confirmed_mask].tolist()
X_reduced = X_topk[confirmed_features].copy()

print("Boruta confirmed:", len(confirmed_features))
print("X_reduced:", X_reduced.shape)

# Save selected features
X_reduced.to_csv("Boruta_Selected_Final_Features.csv", index=False)
print("Saved: Boruta_Selected_Final_Features.csv")


In [ ]:
# Train Tuned RF + Calibrate (Platt)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y_vec, test_size=0.20, stratify=y_vec, random_state=RANDOM_STATE
)

# Tuned RF
rf_tuned = RandomForestClassifier(
    n_estimators=600,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_tuned.fit(X_train, y_train)

# Calibrate (Platt Scaling)
cal_model = CalibratedClassifierCV(estimator=rf_tuned, method="sigmoid", cv=5)
cal_model.fit(X_train, y_train)

print("Calibrated model trained successfully.")
print("Train:", X_train.shape, "| Test:", X_test.shape)


In [ ]:
# Cross-Validated Threshold Optimization
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

thresholds = np.linspace(0.0, 1.0, 200)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_thresholds, fold_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), 1):
    X_tr_cv, X_val_cv = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr_cv, y_val_cv = y_train[tr_idx], y_train[val_idx]

    # Fit fresh RF inside each fold
    rf_cv = RandomForestClassifier(
        n_estimators=600, max_depth=15, min_samples_split=5, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_cv.fit(X_tr_cv, y_tr_cv)

    # Calibrate inside the fold
    cal_cv = CalibratedClassifierCV(estimator=rf_cv, method="sigmoid", cv=3)
    cal_cv.fit(X_tr_cv, y_tr_cv)

    # Predict calibrated probabilities
    y_val_prob = cal_cv.predict_proba(X_val_cv)[:, 1]

    best_acc, best_thr = -1.0, 0.50
    for thr in thresholds:
        acc_thr = accuracy_score(y_val_cv, (y_val_prob >= thr).astype(int))
        if acc_thr > best_acc:
            best_acc, best_thr = acc_thr, thr

    best_thresholds.append(best_thr)
    fold_results.append((fold, best_thr, best_acc))
    print(f"Fold {fold}: Best thr={best_thr:.3f} | ACC={best_acc:.3f}")

ACC_THRESHOLD = round(float(np.mean(best_thresholds)), 3)
print("\n Selected threshold (mean across folds):", ACC_THRESHOLD)
print(" Std of thresholds:", round(float(np.std(best_thresholds)), 4))


In [ ]:
# Final Evaluation (Calibrated RF 0.499 Threshold)
y_prob = cal_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= ACC_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = average_precision_score(y_test, y_prob)
acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
prec    = precision_score(y_test, y_pred)
rec     = recall_score(y_test, y_pred)

print("Decision Threshold:", ACC_THRESHOLD)
print(f"ACC={acc:.3f} | F1={f1:.3f} | PREC={prec:.3f} | REC={rec:.3f} | ROC-AUC={roc_auc:.3f} | PR-AUC={pr_auc:.3f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=3))

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5.2,4.2))
plt.plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.3f}")
plt.plot([0,1],[0,1],'--', lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title(f"ROC — Calibrated RF (n_features={X_reduced.shape[1]})")
plt.legend(); plt.tight_layout(); plt.show()

# PR
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob)
plt.figure(figsize=(5.2,4.2))
plt.plot(rec_curve, prec_curve, lw=2, label=f"PR-AUC={pr_auc:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"PR — Calibrated RF (n_features={X_reduced.shape[1]})")
plt.legend(); plt.tight_layout(); plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=[0,1])
plt.figure(figsize=(4.8,4.0))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", cbar=False,
            xticklabels=["Healthy (0)","RA (1)"], yticklabels=["Healthy (0)","RA (1)"])
plt.ylabel("True"); plt.xlabel("Predicted")
plt.title(f"Confusion Matrix (Thr={ACC_THRESHOLD:.3f})")
plt.tight_layout(); plt.show()


In [ ]:
# 5-Fold Cross-Validation (AUC & ACC per fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_aucs, cv_accs = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_reduced, y_vec), 1):
    X_tr, X_val = X_reduced.iloc[tr_idx], X_reduced.iloc[val_idx]
    y_tr, y_val = y_vec[tr_idx], y_vec[val_idx]

    rf_cv = RandomForestClassifier(
        n_estimators=600, max_depth=15, min_samples_split=5, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ).fit(X_tr, y_tr)

    y_val_prob = rf_cv.predict_proba(X_val)[:, 1]
    y_val_pred = (y_val_prob >= ACC_THRESHOLD).astype(int)
    cv_aucs.append(roc_auc_score(y_val, y_val_prob))
    cv_accs.append(accuracy_score(y_val, y_val_pred))
    print(f"Fold {fold}: AUC={cv_aucs[-1]:.3f} | ACC={cv_accs[-1]:.3f}")

print("\nMean AUC:", f"{np.mean(cv_aucs):.3f} ± {np.std(cv_aucs):.3f}")
print("Mean ACC:", f"{np.mean(cv_accs):.3f} ± {np.std(cv_accs):.3f}")

# Line plots
fold_numbers = np.arange(1, 6)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(fold_numbers, cv_aucs, marker='o', linewidth=2)
plt.ylim(0,1); plt.xlabel("Fold"); plt.ylabel("AUC"); plt.title("AUC Across Folds"); plt.grid(True, ls="--", alpha=0.5)
plt.subplot(1,2,2)
plt.plot(fold_numbers, cv_accs, marker='o', linewidth=2)
plt.ylim(0,1); plt.xlabel("Fold"); plt.ylabel("Accuracy"); plt.title("Accuracy Across Folds"); plt.grid(True, ls="--", alpha=0.5)
plt.tight_layout(); plt.show()

# Boxplot summary
plt.figure(figsize=(6,4))
sns.boxplot(data=[cv_aucs, cv_accs])
plt.xticks([0,1], ["AUC","Accuracy"])
plt.ylabel("Score"); plt.title("CV Performance Distribution")
plt.tight_layout(); plt.show()


In [ ]:
# Cross-Validated ROC Curves (Per Fold + Mean ± Std)
from sklearn.metrics import roc_curve, auc

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tprs, aucs = [], []
mean_fpr = np.linspace(0, 1, 100)

plt.figure(figsize=(6,5))
for i, (tr_idx, val_idx) in enumerate(cv.split(X_reduced, y_vec), 1):
    X_tr, X_val = X_reduced.iloc[tr_idx], X_reduced.iloc[val_idx]
    y_tr, y_val = y_vec[tr_idx], y_vec[val_idx]

    rf_cv = RandomForestClassifier(
        n_estimators=600, max_depth=15, min_samples_split=5, min_samples_leaf=1,
        max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ).fit(X_tr, y_tr)

    y_val_prob = rf_cv.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_val_prob)
    fold_auc = auc(fpr, tpr); aucs.append(fold_auc)
    tprs.append(np.interp(mean_fpr, fpr, tpr)); tprs[-1][0] = 0.0
    plt.plot(fpr, tpr, lw=1.8, label=f"Fold {i} (AUC={fold_auc:.3f})")

plt.plot([0,1],[0,1],'--', color='black', lw=1)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("CV ROC (per fold)")
plt.legend(loc="lower right", fontsize=8); plt.tight_layout(); plt.show()

# Mean ROC ± std
plt.figure(figsize=(6,5))
mean_tpr = np.mean(tprs, axis=0); std_tpr = np.std(tprs, axis=0)
mean_tpr[-1] = 1.0; mean_auc = auc(mean_fpr, mean_tpr); std_auc = np.std(aucs)
plt.plot(mean_fpr, mean_tpr, lw=2.5, label=f"Mean ROC (AUC={mean_auc:.3f} ± {std_auc:.3f})")
plt.fill_between(mean_fpr, np.maximum(mean_tpr-std_tpr,0), np.minimum(mean_tpr+std_tpr,1), alpha=0.2, label="±1 STD")
plt.plot([0,1],[0,1],'--', color='black', lw=1)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Mean ROC Across 5-Fold CV")
plt.legend(loc="lower right"); plt.tight_layout(); plt.show()


In [ ]:
# Calibration Curve + Brier Score
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

y_prob_test = cal_model.predict_proba(X_test)[:, 1]
prob_true, prob_pred = calibration_curve(y_test, y_prob_test, n_bins=10, strategy="quantile")
brier = brier_score_loss(y_test, y_prob_test)
print(f"Brier Score: {brier:.4f}")

plt.figure(figsize=(6,5))
plt.plot(prob_pred, prob_true, marker='o', lw=2, label="Model")
plt.plot([0,1],[0,1],'--', color="black", lw=1, label="Perfect")
plt.xlabel("Predicted Probability"); plt.ylabel("True Probability")
plt.title("Calibration Curve"); plt.legend(); plt.grid(True, ls="--", alpha=0.5)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
sns.histplot(y_prob_test, bins=20, kde=True)
plt.xlabel("Predicted Probability"); plt.ylabel("Count")
plt.title("Distribution of Predicted RA Probabilities")
plt.grid(True, ls="--", alpha=0.4); plt.tight_layout(); plt.show()


In [ ]:
# Biomarkers: RF Importance + Genus/Family + Direction
import re

# Refit RF on ALL data
rf_all = RandomForestClassifier(
    n_estimators=600, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
).fit(X_reduced, y_vec)

feat_importance = pd.Series(rf_all.feature_importances_, index=X_reduced.columns).sort_values(ascending=False)

# Clinical vs microbial detection
meta_clin_cols = [c for c in meta_enc.drop(columns=["label"], errors="ignore").columns if c in X_reduced.columns]
hex_like = re.compile(r"^[0-9a-f]{12,}$", re.IGNORECASE)

def feature_type(name):
    if name in meta_clin_cols: return "Clinical"
    return "Microbial" if hex_like.match(str(name)) else "Clinical"

# Taxonomy lookup (genus → else family → else Unclassified)
def extract_rank(tax_str, rank_code):
    m = re.search(fr"{rank_code}__([^;]+)", str(tax_str) if tax_raw is not None else "")
    if m:
        val = m.group(1).strip()
        if val and val.lower() not in {"na","unclassified","uncultured"}:
            return val
    return None

def tax_label(asv_id):
    if (tax_raw is None) or (str(asv_id) not in (tax_raw.index.astype(str))):
        return "Clinical Feature" if asv_id in meta_clin_cols else "Unclassified"
    tax_str = tax_raw.loc[str(asv_id), "Taxon"]
    g = extract_rank(tax_str, "g")
    if g: return g
    f = extract_rank(tax_str, "f")
    return f if f else "Unclassified"

# Group means for direction (on CLR scale)
mean_RA = X_reduced[y_vec==1].mean(axis=0)
mean_HC = X_reduced[y_vec==0].mean(axis=0)
def direction(name): return "Higher in RA" if mean_RA[name] > mean_HC[name] else "Higher in Healthy"

biomarker_df = pd.DataFrame({
    "Feature": feat_importance.index,
    "Feature_Type": [feature_type(f) for f in feat_importance.index],
    "Genus_or_Family": [tax_label(f) for f in feat_importance.index],
    "Importance": feat_importance.values,
    "Mean_RA": [mean_RA[f] for f in feat_importance.index],
    "Mean_Healthy": [mean_HC[f] for f in feat_importance.index],
    "Direction": [direction(f) for f in feat_importance.index]
}).sort_values("Importance", ascending=False).reset_index(drop=True)

display(biomarker_df.head(25))

# Plots: Top Clinical & Top Microbial
clinical_only  = biomarker_df[biomarker_df["Feature_Type"]=="Clinical"].copy().head(7).sort_values("Importance")
microbial_only = biomarker_df[biomarker_df["Feature_Type"]=="Microbial"].copy().head(15).sort_values("Importance")

plt.figure(figsize=(7,5))
plt.barh(clinical_only["Feature"], clinical_only["Importance"])
plt.xlabel("Importance"); plt.ylabel("Clinical Feature"); plt.title("Top Clinical Features")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,7))
plt.barh(microbial_only["Feature"], microbial_only["Importance"])
plt.xlabel("Importance"); plt.ylabel("Microbial ASV"); plt.title("Top Microbial Features")
plt.tight_layout(); plt.show()

# Save single, full table
biomarker_df.to_csv("Biomarker_Table_Final.csv", index=False)
print("Saved: Biomarker_Table_Final.csv")


In [ ]:
# SHAP Explainability
import shap
import numpy as np
import matplotlib.pyplot as plt
import re

print("Starting SHAP explainability on final tuned Random Forest model...")

# Compute SHAP Values
explainer = shap.TreeExplainer(rf_tuned, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_test)

# Select RA class (positive = 1)
if isinstance(shap_values, list):
    shap_class1 = shap_values[1]
else:
    shap_class1 = shap_values[..., 1]

print(f"SHAP values computed: {shap_class1.shape}")

# Replace ASV IDs with Genus/Family Names
def get_tax_label(asv_id):
    """Return Genus if available, else Family; otherwise original ID."""
    if asv_id not in tax_raw.index:
        return asv_id
    tax_str = str(tax_raw.loc[asv_id, "Taxon"])

    g = re.search(r"g__([^;]+)", tax_str)
    if g and g.group(1) not in ["", "NA", "uncultured", "unclassified"]:
        return g.group(1)

    f = re.search(r"f__([^;]+)", tax_str)
    if f and f.group(1) not in ["", "NA", "uncultured", "unclassified"]:
        return f.group(1)

    return asv_id

# Apply readable names to test features
X_test_named = X_test.copy()
X_test_named.columns = [get_tax_label(c) for c in X_test.columns]
print("ASV IDs replaced with Genus/Family names where available.")

# Global SHAP Summary Plot (Direction + Strength)
plt.figure(figsize=(7, 5))
shap.summary_plot(shap_class1, X_test_named, plot_type="dot", show=False)
plt.title("Global SHAP Summary Plot — RA Class", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig("SHAP_Global_Summary_RA.png", dpi=300)
plt.show()

# SHAP Bar Plot — Top 15 Global Features
plt.figure(figsize=(7, 5))
shap.summary_plot(shap_class1, X_test_named, plot_type="bar", max_display=15, show=False)
plt.title("Top 15 Global SHAP Features (Overall Impact)", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig("SHAP_Top15_Bar_RA.png", dpi=300)
plt.show()

# Save SHAP Values for Reproducibility
np.save("SHAP_values_RF.npy", shap_class1)
print("\n Saved files:")
print("   • SHAP_Global_Summary_RA.png")
print("   • SHAP_Top15_Bar_RA.png")
print("   • SHAP_values_RF.npy")
print(" SHAP Explainability completed successfully.")


In [ ]:
# Final Model Summary & Artifact Export
import json
import joblib
import datetime
from pathlib import Path
# Define Model Identity
model_name = "InteRA-BM: Integrated Clinical–Microbiome Biomarker Discovery and Prediction Model for Rheumatoid Arthritis"
model_version = "v1.0"
creation_date = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

summary_info = {
    "Model_Name": model_name,
    "Version": model_version,
    "Created_On": creation_date,
    "Algorithm": "Random Forest (600 trees, depth=15, sqrt features)",
    "Calibration": "Platt Scaling (Sigmoid, 5-fold)",
    "Feature_Selection": "ExtraTrees + Boruta on CLR-ASV + Clinical",
    "Explainability": "SHAP (Global + Directional + Bar plots)",
    "Novelty": "Combines clinical metadata and CLR-transformed ASVs with explainability layer for RA classification",
    "Performance": {
        "Accuracy": round(acc, 3),
        "F1_Score": round(f1, 3),
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "ROC_AUC": round(roc_auc, 3),
        "PR_AUC": round(pr_auc, 3),
        "Decision_Threshold": ACC_THRESHOLD
    },
    "Files_Saved": [
        "rf_tuned.pkl",
        "cal_model.pkl",
        "Boruta_Selected_Final_Features.csv",
        "Biomarker_Table_Final.csv",
        "SHAP_values_RF.npy",
        "SHAP_Global_Summary_RA.png",
        "SHAP_Top15_Bar_RA.png"
    ]
}


# Create “Artifacts” Folder & Save Everything
save_dir = Path("Artifacts")
save_dir.mkdir(exist_ok=True)

# Save trained models
joblib.dump(rf_tuned, save_dir / "rf_tuned.pkl")
if "cal_model" in locals():
    joblib.dump(cal_model, save_dir / "cal_model.pkl")

# Save key data & documentation
X_reduced.to_csv(save_dir / "Boruta_Selected_Final_Features.csv", index=False)
biomarker_df.to_csv(save_dir / "Biomarker_Table_Final.csv", index=False)
np.save(save_dir / "SHAP_values_RF.npy", shap_class1)

# Move SHAP plots if exist
import shutil
for f in ["SHAP_Global_Summary_RA.png", "SHAP_Top15_Bar_RA.png"]:
    if Path(f).exists():
        shutil.move(f, save_dir / f)

# Save summary info
with open(save_dir / "Model_Summary.json", "w") as f:
    json.dump(summary_info, f, indent=4)

print("\n Files saved in: ./Artifacts/")
for f in summary_info["Files_Saved"]:
    print(f"   • {f}")

# Display Final Summary
print("\n MODEL SUMMARY")
print(f"Model: {model_name}")
print(f"Version: {model_version}")
print(f"Created On: {creation_date}")
print(f"Total Features Used: {X_reduced.shape[1]}")
print("\n--- Performance ---")
for k, v in summary_info["Performance"].items():
    print(f"{k:18s}: {v}")
print("\n Model and artifacts saved successfully.")
